# Prototyping: (Module) Serial Number Reservation for Assembly Sites

In [1]:
import api
import util

In [2]:
import data

In [3]:
manus = data.relevant_manufacturer_IDs_by_shortname
locs = data.relevant_location_IDs_by_shortname
KoPs = data.KoPID_from_partKoPName

get all existing modules, and their SNs first (only those which are non-deleted)

In [4]:
modules, last_responseText = util.get_relevant_parts('Module')

In [5]:
modules

[{'part_id': 35873,
  'is_record_deleted': 'F',
  'record_insertion_time': '2025-10-21T17:58:05Z',
  'record_insertion_user': 'ATLAS_HGTD_PROD',
  'barcode': '',
  'serial_number': 'test_test_AS_2',
  'version': '',
  'name_label': '',
  'installed_date': None,
  'removed_date': None,
  'installed_by_user': '',
  'removed_by_user': '',
  'comment_description': 'test_test',
  'record_lastupdate_time': None,
  'record_lastupdate_user': '',
  'production_date': None,
  'batch_number': '',
  'kind_of_part': {'kind_of_part_id': 1005,
   'display_name': 'Module',
   'is_record_deleted': 'F',
   'record_insertion_time': None,
   'record_insertion_user': 'mrao',
   'record_lastupdate_time': None,
   'record_lastupdate_user': '',
   'comment_description': 'Assembled from 2 hybrids and a module flex'},
  'location': {'location_id': 1521,
   'is_record_deleted': 'F',
   'location_name': 'test_location',
   'record_insertion_time': '2025-08-09T06:43:34Z',
   'record_insertion_user': 'smwang',
   '

In [6]:
last_responseText

'200, OK'

In [7]:
module_SNs = util.get_SN_of_parts(modules)

In [8]:
module_SNs

['test_test_AS_2',
 'test_test_AS',
 '20WMOMT1000001',
 '20WMOFP1000004',
 '20WMOFP1000003',
 '20WMOFP1000002',
 '20WMOFP1000001',
 '98DFMFR065_MAR',
 '98DFMFR063_KUW',
 '99DFMFR065_MAR',
 '99DFMFR063_KUW',
 '99DFMFR065_KUW',
 '99WMOtestYY001',
 '20WMOHT2000008',
 '20WMOHT2000007',
 '20WMOHT2000005',
 '98WMO321000041',
 '98WMO321000045',
 '98WMO321000044',
 '98WMO321000043',
 '98WMO321000042',
 '98WMO321000039',
 '98WMO321000040',
 '98WMO321000038',
 '98WMO321000037',
 '98WMO321000036',
 '98WMO321000035',
 '98WMO321000034',
 '98WMO321000033',
 '98WMO321000032',
 '98WMO321000031',
 'testModule_4_Annika',
 'testModule_3_Annika',
 '98WMO321000023',
 '98WMO321000030',
 '98WMO321000029',
 '98WMO321000028',
 '98WMO321000027',
 '98WMO321000026',
 '98WMO321000025',
 '98WMO321000024',
 '98WMO321000022',
 '98WMO321000021',
 '98WMO321000020',
 'testModule_2_Annika',
 'testModule_1_Annika',
 '99WMO321000014',
 '99WMO321000016',
 '99WMO321000015',
 '99WMO321000013',
 '99WMO321000011',
 '99WMO321000

now need to pattern-match, if we know the production, site, batch, there are six more digits available

In [9]:
# from FADAPro declare_SN.py which uses the specification doc
site_id = {
    'ifae'   : 'F',
    'ihep'   : 'H',
    'ijclab' : 'J',
    'mainz'  : 'M',
    'mascir' : 'A',
    'ustc'   : 'U',
}

prod_id = {
    'prod'    : 'M', # main production
    'preprod' : 'P', # preproduction
    'demo'    : 'D', # demonstrator
    'test'    : 'T', # test
    'other'   : 'O', # other
}

def get_SN_prefix(site, prod, batch):
    k = site_id[site.lower()]
    p = prod_id[prod.lower()]
    b = str(batch)
    if len(b) > 1:
        raise RuntimeError(f'SN : batch should be a single character but you passed {b}')
    if not b.isalnum():
        raise RuntimeError(f'SN : batch should be alphanumeric but you passed {b}')
    snprefix = f'20WMO{k}{p}{b}'
    return snprefix.upper()


In [10]:
my_site = 'mainz'
my_prod = 'test'
my_batch = '1'

In [11]:
my_prefix = get_SN_prefix(my_site, my_prod, my_batch)

In [12]:
my_prefix

'20WMOMT1'

In [13]:
def find_matching_SNs(prefix, existing_SNs):
    return [ex for ex in existing_SNs if str(ex[:8]) == str(prefix)]

In [14]:
find_matching_SNs(my_prefix, module_SNs)

['20WMOMT1000001']

In [15]:
my_matching_SNs = find_matching_SNs('20WMO221', module_SNs)

In [16]:
my_matching_SNs = find_matching_SNs(my_prefix, module_SNs)

In [17]:
my_matching_SNs

['20WMOMT1000001']

In [18]:
def find_matching_sorted_counters(matching_SN_list):
    return sorted([SN[8:] for SN in matching_SN_list])

In [19]:
matched_sorted_counters = find_matching_sorted_counters(my_matching_SNs)

In [20]:
matched_sorted_counters

['000001']

In [21]:
def change_counter_digits_to_ints(counters):
    return [int(c) for c in counters]

In [22]:
matched_sorted_counters_as_INT = change_counter_digits_to_ints(matched_sorted_counters)

In [23]:
matched_sorted_counters_as_INT

[1]

In [24]:
def find_max(my_list):
    return max(my_list) if len(my_list) > 0 else 0

In [25]:
max_matched = find_max(matched_sorted_counters_as_INT)

In [26]:
max_matched

1

In [27]:
def collect_possible_holes(counters_as_INT, max_match):
    holes = []
    # we do not apply zero-counting, because these are human-readable counters starting from 1
    for i in range(1, max_match + 1):
        if i not in counters_as_INT:
            holes.append(i)
    return holes

In [28]:
holes_in_matched = collect_possible_holes(matched_sorted_counters_as_INT, max_matched)

In [29]:
holes_in_matched

[]

In [30]:
how_many_to_reserve = 6

In [31]:
def build_recommended_SNs(N_to_reserve, holes_to_use, max_so_far, prefix):
    n_holes_to_use = len(holes_to_use)
    if n_holes_to_use >= N_to_reserve:
        recom_counters = holes_to_use[:N_to_reserve]
    else:
        still_to_build = N_to_reserve - n_holes_to_use
        recom_counters = holes_to_use + [new_ones for new_ones in range(max_so_far + 1, max_so_far + 1 + still_to_build)]

    return [f'{prefix}{counter:06}' for counter in recom_counters]

In [32]:
recommended_SNs = build_recommended_SNs(how_many_to_reserve, holes_in_matched, max_matched, my_prefix)

In [33]:
recommended_SNs

['20WMOMT1000002',
 '20WMOMT1000003',
 '20WMOMT1000004',
 '20WMOMT1000005',
 '20WMOMT1000006',
 '20WMOMT1000007']

now we know what should be "reserved"

but there is no such thing like "reserving" in the ProdDB, we either record these SNs as parts, or we don't

but for users locally, at a site, it could be useful to keep track of what was "reserved" by them

that is why, we can store a local set of SNs that were generated using hgtd-tools, and second, we will write a comment automatically when the SN part is created in the DB how it was generated

if this is done by users who are really preparing their assembly for the week, also ask them to include the custom "Name Label" while reserving the SNs, this is to ensure we do not miss this crucial mapping step

In [34]:
name_labels = ['aBCdEfg',
               'thisIsATestName',
               'my_test',
               'ojwerpgavroinfgwrosv',
               'whatever',
               'NOTAREALNAME',
              ]

In [35]:
standard_comment_string = 'part SN reserved via hgtd-tools'

```
barcode:this.barcode,
                serial_number:this.serial_no,
                is_record_deleted: 'F',
                version: this.version,
                name_label:this.name_label,
                comment_description: this.comm_desc,
                kind_of_part: digits1,
                location:digits2,
                manufacturer:digits3
```

In [36]:
KoPs['Module']

1005

In [37]:
locs['test']

1521

In [38]:
manus['test']

1161

In [39]:
part_to_upload = {"barcode":"",
                  "serial_number":recommended_SNs[0],
                  "is_record_deleted":"F",
                  "version":"",
                  "name_label":name_labels[0],
                  "comment_description":standard_comment_string,
                  "kind_of_part":str(KoPs['Module']),
                  "location":str(locs['test']),
                  "manufacturer":str(manus['test'])}

In [40]:
last_responseText = api.post_information(
                                    "/partslist", part_to_upload, dryrun=True
                                )

>>> Dryrun post operation with endpoint /partslist
>>> and payload {'barcode': '', 'serial_number': '20WMOMT1000002', 'is_record_deleted': 'F', 'version': '', 'name_label': 'aBCdEfg', 'comment_description': 'part SN reserved via hgtd-tools', 'kind_of_part': '1005', 'location': '1521', 'manufacturer': '1161'}


if you are happy with the dryrun, let's go and upload some test

In [42]:
last_responseText = api.post_information(
                                    "/partslist", part_to_upload, dryrun=False
                                )

HTTPError: [Errno Http Error:] 500 Server Error: Internal Server Error for url: https://hgtddb-api.web.cern.ch/hgtddb/partslist

apparantly, the upload was successful, although we have a 500 error... unclear, continue investigating

https://nginx-hgtddb.app.cern.ch/viewparts/35892